In [ ]:
import pandas as pd

# Load the OBS_SUBJECTS.csv file
obs_subjects = pd.read_csv("obs_SUBJECTS.csv", sep=",", dtype=str)

# Keep only the required columns
obs_subjects = obs_subjects[['usubjid', 'obs_enrolldat']].dropna()

# Convert 'admndat' to datetime
obs_subjects['obs_enrolldat'] = pd.to_datetime(obs_subjects['obs_enrolldat'], errors='coerce')

# Load the follow-up data file
follow_up_data = pd.read_csv("follow-up_obs_DCC_data_release_v2-0-1.tsv", sep="\t", dtype=str)

# Extract 'usubjid' from 'cases.submitter_id' (ensure it remains a string)
follow_up_data['usubjid'] = follow_up_data['cases.submitter_id'].str.extract(r'(\d+)')[0]

# Merge to get the 'admndat' as the baseline date
merged_data = follow_up_data.merge(obs_subjects, on='usubjid', how='left')

# Convert 'admndat' to datetime again for calculations
merged_data['obs_enrolldat'] = pd.to_datetime(merged_data['obs_enrolldat'], errors='coerce')

# Identify columns ending with '_date'
date_columns = [col for col in merged_data.columns if col.endswith('_date')]

# Convert date columns to number of days from baseline and leave empty if NaN
for col in date_columns:
    merged_data[col] = pd.to_datetime(merged_data[col], errors='coerce')  # Convert to datetime
    merged_data[col] = (merged_data[col] - merged_data['obs_enrolldat']).dt.days  # Compute days since baseline
    merged_data[col] = merged_data[col].apply(lambda x: '' if pd.isna(x) else int(x))  # Ensure empty cells instead of 0

# Save the processed file as TSV
merged_data.to_csv("follow-up_obs_DCC_data_release_v2-0-1_indexday.tsv", sep="\t", index=False)

print("Processing complete. The output file is 'processed_obs_follow_up_data.tsv'.")